# Bootstrap the VisDrone benchmark from GitHub

Run this same notebook in Google Colab, Kaggle, or another local Jupyter environment. It detects the host, resolves writable paths, installs only the shared stack, and runs read-only diagnostics. It never starts model training.

In [ ]:
REPOSITORY_URL = "https://github.com/Harryphan72007/aerial-object-detection-benchmark.git"
REPOSITORY_PATH = ""  # Empty selects the platform default.
REFERENCE_TYPE = "branch"  # branch, tag, or commit
REFERENCE = "main"
DRIVE_ROOT = ""  # Empty selects Drive, Kaggle working, or local artifacts.
MOUNT_GOOGLE_DRIVE = True
INSTALL_SHARED_DEPENDENCIES = True


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

repository_default = (
    Path("/content") / "aerial-object-detection-benchmark"
    if Path("/content").is_dir()
    else Path("/kaggle/working") / "aerial-object-detection-benchmark"
    if Path("/kaggle/working").is_dir()
    else Path.cwd()
    if (Path.cwd() / "src" / "notebook_bootstrap.py").is_file()
    else Path.cwd() / "aerial-object-detection-benchmark"
)
repository = (
    Path(REPOSITORY_PATH).expanduser() if REPOSITORY_PATH else repository_default
).resolve()
if not (repository / "src" / "notebook_bootstrap.py").is_file():
    repository.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", REPOSITORY_URL, str(repository)], check=True)
sys.path.insert(0, str(repository))

# checkout_repository_ref validates the checkout itself (linked worktrees
# included) before it fetches or moves HEAD.
from src.colab_setup import checkout_repository_ref
state = checkout_repository_ref(repository, REFERENCE, REFERENCE_TYPE)
print(state)


In [ ]:
from src.notebook_bootstrap import bootstrap_notebook
from src.colab_setup import initialize_drive_directories, validate_drive_writable
from scripts.diagnostics.run_diagnostics import build_report

bootstrap = bootstrap_notebook(
    repository,
    artifact_root=DRIVE_ROOT or None,
    use_google_drive=MOUNT_GOOGLE_DRIVE,
    requirements_file="requirements/legacy-colab.txt",
    install_dependencies=INSTALL_SHARED_DEPENDENCIES,
    require_clean=True,
)
notebook_environment = bootstrap.environment
IN_COLAB = notebook_environment.platform == "colab"
IN_KAGGLE = notebook_environment.platform == "kaggle"
NOTEBOOK_PLATFORM = notebook_environment.platform
DRIVE_ROOT = notebook_environment.artifact_root
paths = initialize_drive_directories(DRIVE_ROOT)
validate_drive_writable(paths.root)
print(bootstrap.summary())
print(build_report(repository))


Bootstrap complete. Continue with `00_prepare_visdrone.ipynb` at the same selected commit. Large artifacts remain under `DRIVE_ROOT`.